# Scientific notes

Definitions and interpretation limits for the catalogue comparisons, transfer tests and Local Group application.

## The question

This thesis investigates the robustness of NPE under cross-simulation transfer of FLAMINGO and TNG300. Using Local Group analogues, posterior distributions of M200c for FLAMINGO are compared against TNG300. Furthermore, a bootstrap-based feature selection framework is provided that identifies observables that improve the robustness of SBI.

Our contribution is diagnostic rather than corrective. The thesis evaluates which catalog-level observables at the Local Group scale are responsible for model misspecification and proposes a bootstrap-based feature ablation framework to diagnose this misspecification.

The main notebook sequence includes both model experiments, the repeated-subsample references, the alternative target, stellar normalization, common support, and the Local Group application. See the [thesis map](thesis_map.ipynb) for the table and figure correspondence.


## Two experiments, two meanings of dynamics

The feature-block experiment contains 15 FLAMINGO models and five TNG300 models. Every block starts with separation, peculiar radial velocity, tangential speed, and the log bound-subhalo mass ratio. FLAMINGO has full-population, same-host, and different-host variants; TNG300 uses the full population because its different-host sample is small.

The final-six experiment has FLAMINGO, TNG300, and balanced combined training, each with two input sets:

| Name used in the final-six experiment | Inputs |
|---|---|
| Dynamics | Separation, peculiar radial velocity, tangential speed |
| Dynamics + stellar sum | Those three inputs plus `log10(Mstar,a + Mstar,b)` in solar-mass units |

The earlier block-grid `core_dynamics` includes the bound mass ratio. The final-six `dynamics` does not. Scores from these models answer different questions and should not be compared as though only the training seed changed.

The log bound mass ratio uses total bound subhalo masses. It is not a stellar ratio and is not used to condition the final observed Local Group posteriors, because it would require an external halo-mass estimate.


## Catalogue selection and mass definitions

Both simulations are analysed at redshift zero. The stellar-mass window is $2\times10^{10}$ to $5\times10^{11}\,M_\odot$. The FLAMINGO selection also requires at least 100 stellar particles. After eligibility, the sample contains a central and the most massive eligible satellite per host where one is available.

A common physical unit system (kpc for distances and M for masses) is enforced across both simulations before any downstream processing. These harmonized features and the unit conventions form the basis for the mass window and the quality cuts, which are used to construct a Local Group analogue sample.

Here, masses are in solar masses; the plain `M` in that extracted passage represents the solar-mass notation in the thesis.

Candidate pairs have a physical separation of 500–1,000 kpc. A same-host pair shares a host FoF ID. A different-host pair has two host IDs. Periodic positions matter for distances and for the mass-weighted pair centre.

Two isolation cuts are applied. The first rejects a pair if another subhalo in either member's host has stellar mass at least $1.5\max(M_{\star,a},M_{\star,b})$. The second rejects a pair if a third subhalo within 2,000 physical kpc of the pair centre has total bound mass greater than $0.5(M_{\mathrm{bound},a}+M_{\mathrm{bound},b})$. These cuts use different masses and different search regions.

The model target is the log of one common host $M_{200c}$ for a same-host pair, or the log of the summed host $M_{200c}$ masses for a different-host pair. It is not the sum of the two galaxy subhalo masses in both cases. The alternative target is the summed bound-subhalo mass, defined consistently across both host classes.

The final populations are 142,713 FLAMINGO pairs and 1,530 corrected TNG300 pairs. The TNG300 catalogue is TNG300-1. The older 1,761-pair TNG selection is superseded.


## Feature equalization

To ensure a fair comparison between the simulations, we recompute stellar velocity dispersion, stellar mass fraction in metals, and κcorot. This is because either the field does not exist in the other simulation or it is computed differently. We choose to convert TNG300 to FLAMINGO equivalent fields because the difference in sample size makes it faster to compute the TNG300 fields from the particle-level data.

The TNG calculations use bound stellar particles, exclude wind particles, and apply no fixed aperture. Stellar dispersion uses the stellar mass-weighted centre-of-mass velocity and the RMS of the diagonal dispersion tensor. Ordered rotation is not subtracted. Kappa uses positions relative to the potential centre and velocities relative to the stellar centre-of-mass velocity. Metallicity is the mass-weighted stellar metal mass fraction, with $Z_\odot=0.0127$ when expressed relative to solar.

These steps harmonize measurement definitions as far as the established workflow allows. They do not make the halo finders, particle membership, or galaxy-formation models identical. Photometry remains the original dust-free catalogue photometry.

Pair-level mean and absolute difference features must be distinguished from member-level distributions. A member contributes once per pair in which it occurs; these are not unique-galaxy statistics.


## Splits, training, and the effective prior

We construct a graph where nodes are host halos and edges connect host-i and host-j. We run connected components on this graph, and two hosts belong to the same component if they can be directly linked by a pair or through a chain of pairs.

Connected components are partitioned into training, validation, and test sets. For example, pairs A–B and B–C must stay together because their hosts are connected through B. A scaler is fitted on training rows only. The final saved splits are:

| Simulation | Training | Validation | Test |
|---|---:|---:|---:|
| FLAMINGO | 109,150 | 12,154 | 21,409 |
| TNG300 | 1,169 | 130 | 231 |

The TNG test contains 183 same-host and 48 different-host pairs. Results on the 48-pair subset have much greater sampling uncertainty than pooled FLAMINGO results.

The established scalar MAF uses five transforms, 64 hidden features, batch size 64, and learning rate 0.00016488693479340492. Training allows up to 200 epochs and stops after 20 epochs without validation improvement, restoring the best validation checkpoint. Optuna tuned the initial dynamics configuration over 30 trials; those hyperparameters were then reused. The historical search included a `num_layers` value that was not passed to the density estimator. The public description keeps this limitation visible.

The scalar affine MAF is a conditional Gaussian in log mass before support restriction. It cannot represent arbitrary skewness or multimodality at a fixed input. Truncation and explicit mixtures can change the shape of the reported distribution.

The declared uniform box from 11.5 to 14.5 bounds rejection sampling. Training target masses came from the selected simulation population, not from uniform draws over that box. The empirical training mass distribution is therefore the effective training prior for these single-round NPE fits.

For combined training, every batch contains 32 uniformly sampled FLAMINGO and 32 TNG300 rows, with replacement. Equal training-domain weights do not imply equal local support at every observed input. The learned combined conditional Gaussian is also not generally the exact arithmetic mixture of the two specialist posteriors.


## Performance and calibration

R2 and RMSE are used in conjunction. This is because R2 depends on the target variance of the evaluation population.

RMSE, signed bias, and interval widths are in dex. Predictions are posterior medians. Signed bias is the mean predicted logarithmic mass minus the true logarithmic mass, so negative bias means underprediction. The feature-block reports use 200 draws per pair; the final-six and follow-up reports use 500.

Coverage and rank statistics diagnose calibration over the evaluated population, but they do not guarantee calibration for every individual posterior.

The reported ranks and coverage are held-out simulation checks, not formal prior-predictive SBC. A rank counts draws below the true mass. Rank-CDF distance measures disagreement with the discrete-uniform reference. Beta is absolute posterior-median error divided by posterior standard deviation; 0.6745 is an ideal Gaussian reference for its median, not a threshold proving calibration. No formal calibration conclusion follows from R2 alone.

Table 5.8 uses the fixed 231 TNG test pairs. Table 5.12 uses all 1,530 TNG pairs for the supporting ablation comparison. Their different populations must remain visible when their coverage numbers are displayed together.


## Bootstrap and sample-size experiments

The population experiment compares TNG-sized FLAMINGO subsamples with the remaining FLAMINGO population, repeating 500 times. The observed cross-simulation mass-distribution discrepancy exceeds all 500 reference values. The corrected empirical probability is $1/501$, approximately 0.002. This reference addresses sample-size reduction within FLAMINGO; it does not identify a physical cause for the simulation difference.

The posterior reference trains a new NPE on 1,530 sampled FLAMINGO pairs in each of 200 repetitions. This includes training stochasticity. The ablation reference evaluates models at the same 500 TNG conditioning rows with 500 posterior draws per row, so a feature change does not also change the evaluation inputs.

The observed core-dynamics JSD is 0.043329 bits, compared with a 200-refit reference median of 0.026353 bits. Fifty refits reach or exceed it, giving the plus-one probability 51/201 = 0.253731. This core-dynamics posterior test does not reject its sample-size/training-variability reference. The population KS result (1/501) and the separate feature-ablation W2 results (1/201 for each tested variant) answer different questions. The thesis §5.7 statement about exceeding all 200 reference values describes the ablation experiment, not the core-dynamics JSD reference.

Saved-prediction component bootstraps answer a different question. They estimate uncertainty in metrics conditional on the already fitted model and saved predictions. Paired contrasts keep component draws aligned across candidate and reference models. They do not propagate all uncertainty from model fitting, input calibration, or the simulator itself.

The notebooks use saved results for the default notebook sequence. Repeating model refits is an optional reproduction step; those expensive experiments are not silently replaced by a bootstrap of aggregate table rows.


## Input diagnostics and ablation

These metrics answer different questions. KS, MMD2, and GMM answer whether the training populations differ; JSD answers whether the posteriors at the observations differ; W2 answers whether the posteriors across a population differ.

MMD uses a pooled fixed bandwidth and 200 label permutations, with corrected minimum probability $1/201$. GMM models and scalers are fitted on FLAMINGO training rows and evaluated on held-out FLAMINGO and TNG rows; scores are given per dimension so feature blocks of different sizes can be compared more fairly. JSD uses normalized, floored densities on a shared grid. W2 is reported on the log-mass scale.

One important thing to consider is that detectable input shifts do not show that the block will fail to transfer successfully.

Likewise, features with similar input distributions do not establish successful transfer.

Removing both variants of stellar velocity dispersion drops the Wasserstein-2 distance to 0.1162 from 0.1887, a 38.4 percent reduction. Removing κcorot does not have a similar effect. This ablation experiment identifies stellar velocity dispersion as a feature that is sensitive to simulation transfer.

The removed variants are the pair mean and absolute difference. The ablation identifies an influential input group, not a unique physical explanation. The whole morphology block's within-simulation gain cannot be attributed to dispersion alone from the saved ablation outputs. A morphology common-support experiment was not performed.


## Stellar normalization and common support

Several competing explanations could produce the observed transfer failure. This section experiments with several competing explanations. However, this does not isolate a physical cause, as several simulation-level differences remain unresolved within the scope of this thesis.

The stellar offset is fitted using training data only. The fixed calibrated value is 0.3616893786 dex, with a 95% component-bootstrap interval of [0.3527423499, 0.3693033411]. It is applied to logarithmic TNG stellar inputs before scaling. Both the individual and summed stellar inputs are evaluated across a fixed shift grid, including zero and the calibrated value.

This result indicates that predictions are sensitive to stellar mass normalizations. ∆cal was fit from training-only mapping and does not correspond to a measured physical error.

The shift experiment uses the earlier specialist models and 231 fixed TNG test rows. Those fits are not the final-six stellar-sum checkpoints. The alternate-target experiment uses all 1,530 TNG pairs. The two panels of thesis Table 5.13 should not be read as a single matched test.

The common-support analysis trains a feature-only histogram gradient-boosted classifier, with FLAMINGO labelled 0 and TNG300 labelled 1. Inverse-frequency weights give the domains equal total weight. Five-fold cross-validation grouped by host component produces out-of-fold scores. The classifier has 160 iterations, learning rate 0.06, 15 leaves, at least 30 samples per leaf, and L2 regularization 1.0.

The principal overlap band is 0.4–0.6. It retains similar classifier scores, not exact equality of all features. Restricting to that band does not restore stellar-input transfer. Results are also reported across band widths. These are diagnostics for dynamics and stellar configurations; they do not test the joint morphology inputs.

Similar marginal distributions do not suggest that joint distributions will be similar.

This framework cannot uniquely separate changes in likelihood from the training prior since both contribute towards the posterior.


## Empirical prior and declared support

The empirical-prior sensitivity uses training-only KDEs on log mass, with Scott bandwidth and factors 0.5, 1, and 2. Saved posterior draws are reweighted by the destination-training density divided by the source-training density. For combined models, the source prior follows equal-domain training. Report effective sample size alongside the reweighted result and do not choose a bandwidth because it supports a preferred interpretation.

Forward dynamics mean bias approaches zero under one prior replacement, but this does not restore individual accuracy. Reverse dynamics is sensitive to bandwidth and can have very low ESS. Stellar-sum transfer biases remain substantial under the tested replacements. The calculation preserves the source effective likelihood and cannot isolate feedback, selection, or a unique physical mechanism.

For Local Group observational mixtures, weights are normalized within each observational component before the original equal component weights are restored. There are only 20 proposals per component, so those updates are less resolved than the nominal 100,000-draw updates. A globally reweighted flattened mixture would be a different calculation.

The supplementary box-bound calculation changes the declared rejection support, not the empirical training prior. It uses analytic conditional-Gaussian quantiles and a matched analytic baseline; small differences from archived 500-draw headline metrics are expected. Widening the original box does not explain the large stellar-sum transfer errors. Narrowing a box can still affect individual tails even when it contains every training mass.


## The Local Group application

The nominal physical inputs are separation 765 kpc, radial velocity −109.3 km/s, tangential speed 82.4 km/s, MW stellar mass $6.08\times10^{10}M_\odot$, and M31 stellar mass $10.3\times10^{10}M_\odot$. The summed stellar input is formed by adding the component linear masses and then taking the logarithm.

The models expect peculiar radial velocity: $v_{r,\mathrm{pec}}=v_{r,\mathrm{phys}}-H_0r$, with $r$ in Mpc. FLAMINGO and the combined model use $H_0=68.1$ km/s/Mpc; TNG uses 67.74. The corresponding nominal inputs are −161.3965 and −161.1211 km/s.

Nominal posterior summaries use 100,000 draws. The uncertainty calculation uses 5,000 observational realizations and 20 conditional draws per realization. Positive quantities use the documented positive distributions; M31 stellar mass uses an asymmetric positive log-space approximation.

An approximate observational input mixture is used rather than exact observational marginalization. The physical inputs are sampled independently. The stellar proxies differ from the stellar mass definitions used in the simulations.

Observed MW disc-plus-bulge stellar mass and the M31 aperture estimate are not the same definition as all-bound simulated stellar mass. The input-shift sensitivity in notebook 12 addresses the effect of this possible mismatch without interpreting its tested offsets as measured corrections.

The final models are trained on mixed host populations with no explicit host-class input. They therefore do not estimate an exclusively two-halo target. The 20 nearest TNG training neighbours to the Local Group are all same-host for both final input configurations. Closeness to the observed inputs is not proof that the relevant physical system has the same host definition.

The conditional maps compare posterior medians across an input grid. Their combined-closeness score is $S=|m_C-m_T|-|m_C-m_F|$. Positive values mean the combined median is closer to FLAMINGO; negative values mean it is closer to TNG. Shared support and small-specialist-gap masks matter. A fraction of uniform grid cells is not a population fraction, and median closeness is not a posterior probability that a simulator is correct.

Literature comparisons preserve the target convention. Lemos et al. (2021) report a posterior peak rather than a median. The van der Marel et al. (2012) timing-argument $M_{200c}$ estimate and the combined dynamical virial-mass estimate are different quantities. Conversion and selection assumptions matter before making a quantitative comparison.

The dynamics-only model is the conservative model. The stellar-sum posteriors add more information but, in turn, degrade transfer performance. The literature values and the posteriors themselves do not establish which simulation is the correct one or any physical cause. No consensus mass is given in this section because of the simulation dependence of the posteriors.


## Classifier-weighted specialist mixtures

Notebook 13 is an extension beyond the thesis. Three constructions should remain separate:

| Construction | What is combined |
|---|---|
| Balanced combined NPE | Training rows from the two simulations |
| Observational-input mixture | Conditional posteriors across noisy observed inputs |
| Classifier-weighted specialist mixture | FLAMINGO and TNG specialist densities at a given input |

A feature-only domain score estimates simulation membership under the classifier's training class weights. It is not a probability that either simulator represents the real Universe. A specialist mixture has the form $w_F(x)q_F(\theta\mid x)+w_T(x)q_T(\theta\mid x)$, with weights summing to one. It should be compared with an equal-weight specialist mixture and the learned combined NPE, using documented class priors, classifier validation, target conventions, and Hubble conversions.

For noisy observations, form the appropriate specialist mixture for each observational realization before averaging over realizations. The existence of a mixture density does not demonstrate improved calibration or settle which simulator is preferable. An extension remains exploratory until its held-out evaluation and provenance are documented.


## What remains future work

The differences change together across the simulations, so the work done in this thesis does not isolate transfer failure to a single mechanism.

The proposed follow-ups include a common halo finder and common stellar apertures, controlled FLAMINGO variants, resolution comparisons, halo-mass-based eligibility, a third simulation, and more flexible final-model density estimators. These are proposed experiments, not completed results in this repository.

FLAMINGO m8–m9 differences include recalibration as well as resolution. Feedback variants can change several calibrated parameters together. Rebuilding a halo-mass-selected sample needs the parent catalogue; the existing stellar-selected cache cannot recover objects already excluded. Matching target mass distributions by construction is not evidence that selection explains the original difference.

The earlier NSF/MDN dynamics comparison does not establish that the scalar-Gaussian limitation is harmless for every final stellar or combined posterior. No morphology common-support test or controlled common-finder experiment is claimed.

Another caveat to these experiments is that we do not know which of these simulations represent the real truth. Either one or neither could explain the real relationship between galaxy properties and halo mass in the real universe. This makes our results simulation-dependent. Agreement within a simulation or across simulations or even by using a combined model does not by itself demonstrate accuracy for the real Local Group.


## Notes on the thesis tables

The thesis is the reference for scope and writing. Its saved tables and the underlying saved results take precedence where nearby prose conflicts with them.

| Location | Correction used here |
|---|---|
| §3.6 volume and density prose | TNG side length is $205/h=302.62769$ Mpc, so volume is $2.771571\times10^7\,\mathrm{Mpc}^3$. FLAMINGO's selected pair density is approximately 2.59 times TNG's. The correct Table 3.6 values replace the conflicting nearby prose. |
| §5.7 alternative-target stellar-ratio RMSE | Use the saved Table 5.13 value 0.3199 dex, consistent with the paired difference −0.0354 from dynamics 0.3553. The nearby 0.3119 value is inconsistent. |
| §3.5 central/satellite stellar offsets | The archived binned relation does not verify those subgroup offsets. A clearly defined aggregate statistic is a TNG-minus-FLAMINGO median offset of −0.295877 dex across 13 mass bins with at least 20 members from each simulation. Different regressions and populations must be identified separately. |
| §5.7 sample-size interpretation | The population KS and feature-ablation tests exceed their reference values, but the core-dynamics posterior JSD has plus-one p = 51/201, approximately 0.254. Do not generalize the ablation result to that posterior diagnostic. |
| §4.1 splitting explanation | The fixed design includes training, validation, and test partitions, with checkpoint selection based on validation loss. |
| Scalar-MAF wording | The conditional density is Gaussian before support truncation. Truncated distributions and explicit mixtures need not remain Gaussian. |
| Host-target description in §6.4 | The fitted target mixes one common-host mass and the sum of two host masses, rather than being an exclusively summed two-halo target. |
| Disc fractions and pair-member counts | Use pair-member-weighted fractions 0.284624 for FLAMINGO and 0.459477 for TNG at kappa > 0.35. These do not identify a feedback cause. |

The supplied PDF's conclusions chapter is empty. The repository summarizes the documented results and limitations without attributing a new conclusion to that chapter.
